# ELECTRE I kernel: from pairwise comparisons to a choice set

The outranking matrix is not a ranking. This lab checks the two kernel conditions: internal stability and external stability. The implementation enumerates every kernel for small graphs and reports when none or more than one exists. Read the [method lesson](../methods/electre-outranking.md), [package contract](../implementations/python/electre-i/README.md) and [kernel exercise](../exercises/electre-i-kernel.md).

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, Circle

current = Path.cwd().resolve()
repo_root = next((p for p in (current, *current.parents)
                  if (p / 'implementations' / 'python' / 'electre-i').is_dir()), None)
if repo_root is None:
    raise FileNotFoundError('Run this notebook from a checkout of geo-dev-hub/spatial-analysis.')
package = repo_root / 'implementations' / 'python' / 'electre-i'
sys.path.insert(0, str(package))
from electre_i_kernel import enumerate_electre_kernels


## 1. Start with a small outranking graph

An arrow from A to B means A outranks B. It does not mean that B is reachable from A through several steps; the kernel's external-stability condition uses direct outranking.

In [ ]:
names = ['A', 'B', 'C']
relation = np.array([[0, 1, 0],
                     [0, 0, 1],
                     [0, 0, 0]], dtype=bool)
result = enumerate_electre_kernels(relation, names)
print('Kernels:', result.kernels)
print('Unique:', result.is_unique)


In [ ]:
positions = {'A': (0, 0), 'B': (1.5, 0), 'C': (3, 0)}
fig, ax = plt.subplots(figsize=(8, 2.8))
kernel_members = set(result.kernels[0]) if result.is_unique else set()
for name, (x, y) in positions.items():
    face = '0.85' if name in kernel_members else 'white'
    ax.add_patch(Circle((x, y), 0.28, facecolor=face, edgecolor='black', linewidth=1.5))
    ax.text(x, y, name, ha='center', va='center', fontsize=12)
for i, a in enumerate(names):
    for j, b in enumerate(names):
        if relation[i, j]:
            ax.add_patch(FancyArrowPatch((positions[a][0] + .3, 0),
                                         (positions[b][0] - .3, 0),
                                         arrowstyle='-|>', mutation_scale=14,
                                         connectionstyle='arc3,rad=0.0', color='black'))
ax.set_xlim(-.5, 3.5); ax.set_ylim(-.6, .6); ax.axis('off')
ax.set_title('A → B → C; the kernel is {A, C}')
plt.show()


The kernel is {A, C}. It is internally stable because neither selected alternative outranks the other. It is externally stable because the only excluded alternative, B, is directly outranked by A. The path B → C does not make C outrank B.

## 2. Cycles can change the answer

A directed cycle may prevent a kernel from existing. The implementation does not silently delete edges or apply a cycle-reduction heuristic.

In [ ]:
cycle = np.array([[0, 1, 0],
                  [0, 0, 1],
                  [1, 0, 0]], dtype=bool)
cycle_result = enumerate_electre_kernels(cycle, names)
print('Directed cycle kernels:', cycle_result.kernels)
print('Has a kernel:', cycle_result.has_kernel)


## 3. Multiple kernels are possible

When two alternatives mutually outrank one another, either singleton can satisfy the two kernel conditions. Report the multiplicity; do not choose one by array order.

In [ ]:
mutual = np.array([[0, 1], [1, 0]], dtype=bool)
multiple = enumerate_electre_kernels(mutual, ['North', 'South'])
print('All kernels:', multiple.kernels)
print('Unique:', multiple.is_unique)


## Investigate

1. Why is external stability based on a direct outranking edge?
2. What does it mean if the relation has no kernel?
3. Why can multiple kernels be a meaningful result?
4. How would changing the concordance or discordance thresholds change the graph before kernel extraction?
5. Why is the exhaustive implementation limited to small candidate sets?

Complete the [kernel exercise](../exercises/electre-i-kernel.md). A kernel is a choice set, not a complete ranking.